# ClauseRisk — Expériences complémentaires pour la révision (Round 1)


| Section | Demande | API OpenAI ? |
|---|---|---|
| 5 | Baselines classiques ré-entraînées avec un protocole propre (split par entrée 70/15/15) → Tableau 12 | Non |
| 6 | **XLM-R fine-tuné** (3 seeds) — R1, R2, R4 | Non (GPU) |
| 7 | **Retrieval sans génération** (multilingual-E5 k-NN) — R1, R2, R4 | Non |
| 8 | BM25 + Rule (binaire) | Non |
| 9 | **GPT-4o sans RAG** (LLM-only) — R1, R4 | Oui |
| 10 | **Runs répétés** du pipeline RAG (R = 3) + coût par passage — R2, R3,
| 11 | Comparaison **à couverture égale** + McNemar — R1 | Non |
| 12 | Lignes LaTeX prêtes à coller + export | Non |



In [1]:
!pip install -q openai sentence-transformers transformers accelerate rank_bm25 openpyxl scikit-learn scipy
print("OK")

OK


In [2]:
# ---------------- CONFIGURATION ----------------
SEED = 42
RUN_CLASSICAL   = True
RUN_XLMR        = True    # ~10-15 min sur T4
RUN_E5_KNN      = True
RUN_BM25_RULE   = True
RUN_LLM_ONLY    = True    # appels API GPT-4o
RUN_RAG_REPEATS = True    # appels API GPT-4o (le plus coûteux)
N_REPEATS       = 3
MAIN_MODEL, AUX_MODEL = "gpt-4o", "gpt-4o-mini"
# Prix en USD par million de tokens (à vérifier sur la page tarifs OpenAI au moment du run)
PRICES = {"gpt-4o": (2.50, 10.00), "gpt-4o-mini": (0.15, 0.60)}
XLMR_SEEDS = [42, 43, 44]

import os, re, json, math, time, random, unicodedata, warnings, itertools, ast
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, cohen_kappa_score
from scipy.stats import binomtest
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)
CLASSES = ["low", "medium", "high"]
SEV = {"none": 0, "low": 1, "medium": 2, "high": 3}
OUT = "revision_experiments"; os.makedirs(OUT, exist_ok=True)
RESULTS = {}      # nom -> dict(internal_macro, external_macro, external_binary, preds3, conf)
print("Config OK")

Config OK


In [3]:
# ---------------- UPLOAD ----------------
# Uploader : kb_compact.json, EVAL_external_ENRICHED_MASSIVE.xlsx
# Optionnel : clauserisk_v3_results.xlsx (votes stockés -> v5 pour McNemar et couverture égale)
try:
    from google.colab import files
    up = files.upload(); print(list(up))
except ImportError:
    print("Hors Colab : placer les fichiers dans le dossier courant")
KB_FILE   = next(f for f in os.listdir(".") if f.startswith("kb_compact") and f.endswith(".json"))
EVAL_FILE = next(f for f in os.listdir(".") if f.startswith("EVAL_external") and f.endswith(".xlsx"))
V3_FILE   = next((f for f in os.listdir(".") if f.startswith("clauserisk_v3_results") and f.endswith(".xlsx")), None)
print(KB_FILE, EVAL_FILE, V3_FILE)

Saving clauserisk_v3_results.xlsx to clauserisk_v3_results.xlsx
Saving EVAL_external_ENRICHED_MASSIVE.xlsx to EVAL_external_ENRICHED_MASSIVE.xlsx
Saving kb_compact.json to kb_compact.json
['clauserisk_v3_results.xlsx', 'EVAL_external_ENRICHED_MASSIVE.xlsx', 'kb_compact.json']
kb_compact.json EVAL_external_ENRICHED_MASSIVE.xlsx clauserisk_v3_results.xlsx


In [4]:
# ---------------- DONNÉES ----------------
kb = pd.DataFrame(json.load(open(KB_FILE, encoding="utf-8")))
kb["rl"] = kb["rl"].astype(str).str.lower().str.strip()
kb = kb[kb["rl"].isin(CLASSES)].reset_index(drop=True)
ev = pd.read_excel(EVAL_FILE, sheet_name="passages")
ev = ev[ev["passage_text"].notna() & ev["expected_risk_level"].notna()].copy()
ev["gold"] = ev["expected_risk_level"].astype(str).str.lower().str.strip()
ev = ev[ev["gold"].isin(CLASSES)].reset_index(drop=True)
ev["language"] = ev["language"].astype(str).str.lower()
y_ext3 = ev["gold"].values
y_extb = (y_ext3 == "high").astype(int)           # RISKY = HIGH (Tableau de correspondance)
print(f"KB : {len(kb)} entrées x 3 langues | EVAL-EEM : {len(ev)} passages")
print(kb["rl"].value_counts().to_dict(), "|", pd.Series(y_ext3).value_counts().to_dict())

# Split au niveau de l'ENTRÉE (les 3 langues d'une entrée restent dans le même split)
idx = np.arange(len(kb))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=kb["rl"], random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=kb["rl"].values[tmp_idx], random_state=SEED)
def expand(ids):
    rows = []
    for i in ids:
        r = kb.iloc[i]
        for lg in ["en", "fr", "ar"]:
            t = r.get(lg)
            if isinstance(t, str) and t.strip():
                rows.append({"entry": r["id"], "lang": lg, "text": t, "label": r["rl"]})
    return pd.DataFrame(rows)
TR, VA, TE = expand(tr_idx), expand(va_idx), expand(te_idx)
ALL = expand(idx)
print(f"Lignes train/val/test internes : {len(TR)}/{len(VA)}/{len(TE)} (entrées {len(tr_idx)}/{len(va_idx)}/{len(te_idx)})")

AR_DIAC = re.compile(r"[\u064B-\u065F\u0670\u0640]")
def norm(t, lg=None):
    t = str(t).lower()
    t = AR_DIAC.sub("", t)
    t = re.sub("[إأآ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn" or "\u0600" <= c <= "\u06FF")
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()
ev["norm"] = [norm(t) for t in ev["passage_text"]]

KB : 315 entrées x 3 langues | EVAL-EEM : 81 passages
{'low': 203, 'medium': 64, 'high': 48} | {'high': 37, 'medium': 26, 'low': 18}
Lignes train/val/test internes : 660/141/144 (entrées 220/47/48)


In [5]:
# ---------------- MÉTRIQUES ----------------
def macro3(y, p):
    return f1_score(y, p, labels=CLASSES, average="macro", zero_division=0)
def to_bin(p3):
    return (np.asarray(p3) == "high").astype(int)
def binf1(yb, pb):
    return f1_score(yb, pb, zero_division=0)
def boot_ci(yb, pb, n=2000, seed=SEED):
    rng = np.random.default_rng(seed); pos = np.where(yb == 1)[0]; neg = np.where(yb == 0)[0]; v = []
    for _ in range(n):
        b = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        v.append(f1_score(yb[b], pb[b], zero_division=0))
    return np.percentile(v, 2.5), np.percentile(v, 97.5)
def mcnemar(y, pa, pb):
    a, b = pa == y, pb == y
    n01, n10 = int(np.sum(a & ~b)), int(np.sum(~a & b))
    return n01, n10, (binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0)
def record(name, internal=None, preds3=None, predb=None, conf=None, extra=None):
    predb = to_bin(preds3) if predb is None else np.asarray(predb)
    RESULTS[name] = {"internal_macro": internal,
                     "external_macro": macro3(y_ext3, preds3) if preds3 is not None else None,
                     "external_binary": binf1(y_extb, predb), "preds3": preds3, "predb": predb,
                     "conf": conf, **(extra or {})}
    r = RESULTS[name]
    print(f"{name:<34} int={r['internal_macro'] if r['internal_macro'] is None else round(r['internal_macro'],3)}"
          f"  ext3={None if r['external_macro'] is None else round(r['external_macro'],3)}  extB={r['external_binary']:.3f}")
print("OK")

OK


In [6]:
# ---------------- 5. BASELINES CLASSIQUES + MiniLM ----------------
if RUN_CLASSICAL:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    rng = np.random.default_rng(SEED)
    maj = TR["label"].value_counts().idxmax()
    record("Majority", macro3(TE["label"], [maj]*len(TE)), np.array([maj]*len(ev)))
    record("Random", macro3(TE["label"], rng.choice(CLASSES, len(TE))), rng.choice(CLASSES, len(ev)))
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    Xtr = vec.fit_transform([norm(t) for t in TR["text"]]); Xte = vec.transform([norm(t) for t in TE["text"]])
    Xev = vec.transform(ev["norm"])
    models = {"TF-IDF + LogReg": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
              "TF-IDF + SVM": LinearSVC(class_weight="balanced", random_state=SEED, max_iter=5000),
              "TF-IDF + RF": RandomForestClassifier(200, class_weight="balanced", random_state=SEED),
              "TF-IDF + GB": GradientBoostingClassifier(n_estimators=100, random_state=SEED)}
    for n, m in models.items():
        m.fit(Xtr, TR["label"]); record(n, macro3(TE["label"], m.predict(Xte)), m.predict(Xev))
    from sentence_transformers import SentenceTransformer
    mini = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    Etr, Ete, Eev = (mini.encode(list(x), batch_size=32, normalize_embeddings=True)
                     for x in (TR["text"], TE["text"], ev["passage_text"].astype(str)))
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(Etr, TR["label"])
    pe = lr.predict_proba(Eev)
    record("MiniLM embeddings + LogReg", macro3(TE["label"], lr.predict(Ete)), lr.predict(Eev),
           conf=pe.max(1))

Majority                           int=0.262  ext3=0.121  extB=0.000
Random                             int=0.333  ext3=0.338  extB=0.394
TF-IDF + LogReg                    int=0.809  ext3=0.179  extB=0.000
TF-IDF + SVM                       int=0.786  ext3=0.179  extB=0.000
TF-IDF + RF                        int=0.767  ext3=0.126  extB=0.000
TF-IDF + GB                        int=0.744  ext3=0.147  extB=0.000


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM embeddings + LogReg         int=0.751  ext3=0.25  extB=0.100


In [7]:
# ---------------- 6. XLM-R FINE-TUNÉ ----------------
if RUN_XLMR:
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device", dev)
    L2I = {c: i for i, c in enumerate(CLASSES)}
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
    def batches(texts, labels=None, bs=16, shuffle=False, maxlen=128):
        order = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
        for s in range(0, len(texts), bs):
            ii = order[s:s+bs]
            enc = tok([texts[i] for i in ii], truncation=True, max_length=maxlen, padding=True, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            yield enc, (torch.tensor([L2I[labels[i]] for i in ii]).to(dev) if labels is not None else None)
    def predict_proba(model, texts, maxlen):
        model.eval(); out = []
        with torch.no_grad():
            for enc, _ in batches(texts, bs=32, maxlen=maxlen):
                out.append(torch.softmax(model(**enc).logits, -1).cpu().numpy())
        return np.vstack(out)
    trX, trY = list(TR["text"]), list(TR["label"]); vaX, vaY = list(VA["text"]), list(VA["label"])
    cw = torch.tensor([len(trY) / (3 * max(1, trY.count(c))) for c in CLASSES], dtype=torch.float).to(dev)
    seed_int, seed_ext, probs_ext = [], [], []
    for sd in XLMR_SEEDS:
        torch.manual_seed(sd); np.random.seed(sd); random.seed(sd)
        model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3).to(dev)
        EPOCHS = 10; opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        steps = EPOCHS * math.ceil(len(trX) / 16)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        lossf = torch.nn.CrossEntropyLoss(weight=cw); best, best_state = -1, None
        for ep in range(EPOCHS):
            model.train()
            for enc, yb_ in batches(trX, trY, shuffle=True):
                loss = lossf(model(**enc).logits, yb_); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
            f = macro3(vaY, [CLASSES[i] for i in predict_proba(model, vaX, 128).argmax(1)])
            if f > best: best, best_state = f, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        pi = predict_proba(model, list(TE["text"]), 128); pe = predict_proba(model, list(ev["passage_text"].astype(str)), 256)
        seed_int.append(macro3(TE["label"], [CLASSES[i] for i in pi.argmax(1)]))
        seed_ext.append(macro3(y_ext3, [CLASSES[i] for i in pe.argmax(1)])); probs_ext.append(pe)
        print(f"seed {sd}: val={best:.3f} internal={seed_int[-1]:.3f} external={seed_ext[-1]:.3f}")
        del model; torch.cuda.empty_cache()
    pm = np.mean(probs_ext, 0); p3 = np.array([CLASSES[i] for i in pm.argmax(1)])
    seed_bin = [binf1(y_extb, to_bin([CLASSES[i] for i in p.argmax(1)])) for p in probs_ext]
    record("XLM-R-base, fine-tuned", float(np.mean(seed_int)), p3, conf=pm.max(1),
           extra={"seed_internal": seed_int, "seed_external": seed_ext, "seed_binary": seed_bin})
    print(f"XLM-R external macro-F1 {np.mean(seed_ext):.3f} ± {np.std(seed_ext):.3f} | binary {np.mean(seed_bin):.3f} ± {np.std(seed_bin):.3f}")

device cuda


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 43: val=0.818 internal=0.872 external=0.222


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 44: val=0.807 internal=0.807 external=0.183
XLM-R-base, fine-tuned             int=0.865  ext3=0.224  extB=0.000
XLM-R external macro-F1 0.207 ± 0.017 | binary 0.033 ± 0.047


In [8]:
# ---------------- 7. mE5 k-NN (retrieval sans génération) ----------------
if RUN_E5_KNN:
    from sentence_transformers import SentenceTransformer
    e5 = SentenceTransformer("intfloat/multilingual-e5-base")
    enc_p = lambda xs: e5.encode(["passage: " + x for x in xs], batch_size=32, normalize_embeddings=True)
    enc_q = lambda xs: e5.encode(["query: " + x for x in xs], batch_size=32, normalize_embeddings=True)
    def knn(index_emb, index_lab, q_emb, k=10):
        S = q_emb @ index_emb.T; preds, conf = [], []
        for row in S:
            top = np.argsort(-row)[:k]; sc = defaultdict(float)
            for j in top: sc[index_lab[j]] += max(row[j], 0)
            lab = max(sc, key=sc.get); preds.append(lab); conf.append(sc[lab] / (sum(sc.values()) + 1e-9))
        return np.array(preds), np.array(conf)
    p_int, _ = knn(enc_p(list(TR["text"])), TR["label"].values, enc_q(list(TE["text"])))
    p_ext, c_ext = knn(enc_p(list(ALL["text"])), ALL["label"].values, enc_q(list(ev["passage_text"].astype(str))))
    record("mE5 k-NN (no generation)", None, p_ext, conf=c_ext, extra={"internal_note": macro3(TE["label"], p_int)})
    print("(k-NN interne, index = train uniquement :", round(macro3(TE["label"], p_int), 3), ")")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

mE5 k-NN (no generation)           int=None  ext3=0.17  extB=0.053
(k-NN interne, index = train uniquement : 0.747 )


In [9]:
# ---------------- 8. BM25 + Rule ----------------
if RUN_BM25_RULE:
    from rank_bm25 import BM25Okapi
    bm = {lg: BM25Okapi([norm(t).split() for t in kb[lg].astype(str)], k1=1.5, b=0.75) for lg in ["en", "fr", "ar"]}
    preds = []
    for t, lg in zip(ev["passage_text"].astype(str), ev["language"]):
        sc = bm.get(lg, bm["en"]).get_scores(norm(t).split()); top = np.argsort(-sc)[:5]
        preds.append(max((kb["rl"].iloc[j] for j in top), key=lambda c: SEV[c]))
    record("BM25 + Rule", None, np.array(preds))

BM25 + Rule                        int=None  ext3=0.236  extB=0.388


In [10]:
# ---------------- 9. OPENAI : configuration commune ----------------
if RUN_LLM_ONLY or RUN_RAG_REPEATS:
    from getpass import getpass
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY") or getpass("Clé OpenAI (masquée) : "))
    USAGE = defaultdict(lambda: {"in": 0, "out": 0, "calls": 0, "sec": 0.0})

    VOTE_CFG = [(0.0, "legal"), (0.3, "legal"), (0.5, "consumer"), (0.5, "privacy"), (0.7, "legal")]
    PERSPECTIVE = {"legal": "Reason as an Algerian legal practitioner applying the statutes literally.",
                   "consumer": "Reason from the standpoint of consumer protection (Loi 09-03).",
                   "privacy": "Reason from the standpoint of personal-data protection (Loi 18-07)."}
    def system_prompt(lang, persp):
        ln = {"en": "English", "fr": "French", "ar": "Arabic"}.get(lang, "English")
        return f"""You are an expert multilingual legal risk analyst specializing in Algerian law (Loi 18-07 data protection, Loi 09-03 consumer protection, Loi 18-05 e-commerce, Penal Code).
{PERSPECTIVE[persp]}
CLASSIFICATION RULES:
- HIGH: direct violation of Algerian law OR explicit waiver of rights OR international data transfer without authorization of the national authority OR sensitive/biometric data without explicit consent OR abusive clause forbidden by Loi 09-03 OR restriction of access to Algerian courts
- MEDIUM: potential violation, ambiguous clause, insufficient transparency, unilateral modification
- LOW: standard compliant clause or purely descriptive content
- NONE: no legal content
When a clause engages several provisions at once, prefer HIGH (precautionary principle).
The clause is in {ln}. Output ONLY valid JSON."""

    def chat(model, messages, temperature, max_tokens=60, json_mode=True):
        for attempt in range(4):
            try:
                t0 = time.time()
                kw = {"response_format": {"type": "json_object"}} if json_mode else {}
                r = client.chat.completions.create(model=model, messages=messages, temperature=temperature,
                                                   max_tokens=max_tokens, **kw)
                u = USAGE[model]; u["in"] += r.usage.prompt_tokens; u["out"] += r.usage.completion_tokens
                u["calls"] += 1; u["sec"] += time.time() - t0
                return r.choices[0].message.content
            except Exception as e:
                print("  retry", attempt, str(e)[:120]); time.sleep(2 ** attempt)
        return ""
    def parse_risk(txt):
        try: v = str(json.loads(txt).get("overall_risk", "low")).lower().strip()
        except Exception: v = "low"
        return v if v in SEV else "low"

    # Keyword veto : copie exacte des 25 motifs du notebook v5
    HIGH_KEYWORDS = {
        "fr": [r"\brenonc(?:e|ez|er|iation)\s+(?:irr[ée]vocablement|à\s+tout\s+droit)",
               r"exclusivement\s+devant\s+(?:un|le|les)\s+tribunal", r"d[ée]gag[eé]?\s+toute\s+(?:sa\s+)?responsabilit[ée]",
               r"transf[ée]r[ée]s?\s+(?:vers|en|hors)\s+(?:la\s+)?(?:R[ée]publique\s+populaire\s+de\s+)?Chine",
               r"lois?\s+(?:de\s+l['\u2019])?(?:[ée]tat\s+de\s+)?Californie", r"r[ée]gis?\s+par\s+les\s+lois\s+de\s+Singapour",
               r"reconnaissance\s+faciale", r"donn[ée]es\s+biom[ée]triques", r"cession\s+(?:à\s+titre\s+)?on[ée]reuse?"],
        "en": [r"\birrevocably\s+waive", r"exclusively\s+before\s+(?:a|the)\s+court",
               r"governed\s+by\s+the\s+laws\s+of\s+(?:Singapore|California|the\s+Northern\s+District)",
               r"transferred\s+to\s+(?:and\s+)?(?:stored|processed)\s+(?:in|at).*China",
               r"People['\u2019]?s\s+Republic\s+of\s+China", r"facial\s+recognition", r"biometric\s+(?:data|information)",
               r"binding\s+arbitration", r"(?:perpetual|irrevocable)\s+(?:.*)?(?:licens|royalty-free)", r"sublicensable\s+and\s+transferable"],
        "ar": [r"تتنازل\s+عن", r"محاكم\s+ولاية\s+كاليفورنيا", r"قوانين\s+سنغافورة", r"جمهورية\s+الصين", r"بيانات\s+بيومترية", r"التعرف\s+على\s+الوجه"]}
    def keyword_veto(text, lg):
        pats = HIGH_KEYWORDS.get(lg, []) + HIGH_KEYWORDS["en"]
        return any(re.search(p, str(text).lower(), re.IGNORECASE) for p in pats)
    KW = np.array([keyword_veto(t, l) for t, l in zip(ev["passage_text"], ev["language"])])

    def aggregate(votes_list, k=1, use_kw=True):
        """3 classes : majorité pondérée (1er vote x1.5, égalité -> plus grave) ; binaire : >=k votes HIGH (+veto) ;
        confiance = Eq. (2) : part des votes en accord avec le label émis."""
        p3, pb, conf = [], [], []
        for i, v in enumerate(votes_list):
            sc = defaultdict(float)
            for j, x in enumerate(v): sc[x] += 1.5 if j == 0 else 1.0
            best = max(sc.values()); lab = max([x for x in sc if sc[x] == best], key=lambda x: SEV[x])
            p3.append("low" if lab == "none" else lab)
            nh = sum(x == "high" for x in v); b = int(nh >= k or (use_kw and KW[i]))
            pb.append(b); conf.append(nh / len(v) if b else 1 - nh / len(v))
        return np.array(p3), np.array(pb), np.array(conf)
    def cost(model):
        u = USAGE[model]; pi, po = PRICES[model]; return (u["in"] * pi + u["out"] * po) / 1e6
    print("OpenAI prêt |", KW.sum(), "passages déclenchent le veto")

Clé OpenAI (masquée) : ··········
OpenAI prêt | 15 passages déclenchent le veto


In [11]:
# ---------------- 9b. GPT-4o SANS RAG ----------------
if RUN_LLM_ONLY:
    llm_votes = []
    for t, lg in zip(ev["passage_text"].astype(str), ev["language"]):
        v = []
        for temp, persp in VOTE_CFG:
            msg = [{"role": "system", "content": system_prompt(lg, persp)},
                   {"role": "user", "content": f"Clause:\n{t[:1500]}\n\nOutput ONLY: {{\"overall_risk\": \"none|low|medium|high\"}}"}]
            v.append(parse_risk(chat(MAIN_MODEL, msg, temp)))
        llm_votes.append(v)
    p3, pb, cf = aggregate(llm_votes, k=1, use_kw=True)
    record("GPT-4o without retrieval", None, p3, predb=pb, conf=cf)
    _, pb3, _ = aggregate(llm_votes, k=3, use_kw=False)
    print("  variante >=3 votes sans veto : binary F1 =", round(binf1(y_extb, pb3), 3))
    pd.DataFrame({"passage_id": ev["passage_id"], "votes": [str(v) for v in llm_votes]}).to_excel(f"{OUT}/llm_only_votes.xlsx", index=False)

GPT-4o without retrieval           int=None  ext3=0.746  extB=0.775
  variante >=3 votes sans veto : binary F1 = 0.767


In [12]:
# ---------------- 10. RUNS RÉPÉTÉS DU PIPELINE RAG ----------------
# ATTENTION : ré-implémentation à partir du code v2 (BM25 par langue + dense MiniLM sur entrées trilingues, RRF,
# HyDE, 5 exemples few-shot) avec les 5 votes GPT-4o de v3. Pas de cross-encoder ni de vérificateur.
# Si vous avez le notebook v3 original, relancez-le plutôt R fois et sauvegardez les votes au même format.
if RUN_RAG_REPEATS:
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer
    mini = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    kb_rec = kb.to_dict("records")
    kb_emb = mini.encode([" ".join(str(r.get(l, "")) for l in ["en", "fr", "ar"]) for r in kb_rec], normalize_embeddings=True)
    bm = {lg: BM25Okapi([norm(r.get(lg, "")).split() for r in kb_rec]) for lg in ["en", "fr", "ar"]}
    def retrieve(q, lg, k=10):
        s1 = bm.get(lg, bm["en"]).get_scores(norm(q).split()); r1 = np.argsort(-s1)[:20]
        s2 = kb_emb @ mini.encode([q], normalize_embeddings=True)[0]; r2 = np.argsort(-s2)[:20]
        sc = defaultdict(float)
        for rank, i in enumerate(r1): sc[i] += 1 / (60 + rank + 1)
        for rank, i in enumerate(r2): sc[i] += 1 / (60 + rank + 1)
        return [kb_rec[i] for i, _ in sorted(sc.items(), key=lambda x: -x[1])[:k]]
    FEW = []
    for lab, n in [("high", 2), ("medium", 2), ("low", 1)]:
        for _, c in kb[kb["rl"] == lab].head(n).iterrows():
            FEW.append(f"Clause: {c['en'][:200]}\nReasoning: Violation: {c['rc']} - Legal basis: {str(c['lb'])[:80]}\nOutput: {{\"overall_risk\": \"{lab}\"}}")
    FEW_BLOCK = "EXAMPLES:\n\n" + "\n\n".join(FEW)
    def hyde(t, lg):
        m = [{"role": "user", "content": f"Write a SHORT hypothetical legal clause (1-2 sentences) describing the same type of risk as this clause. Output only the clause.\n\nClause: {t[:500]}"}]
        return chat(AUX_MODEL, m, 0.3, max_tokens=80, json_mode=False)
    rag_runs = []
    U0 = {m: dict(USAGE[m]) for m in list(USAGE)}
    for run in range(N_REPEATS):
        votes, t_start = [], time.time()
        for t, lg in zip(ev["passage_text"].astype(str), ev["language"]):
            ctx = retrieve(t + " " + hyde(t, lg), lg)
            ctx_txt = "\n".join(f"[{c['id']}] (risk={c['rl']}, cat={c['rc'][:40]}): {(c.get(lg) or c['en'])[:200]}" for c in ctx)
            v = []
            for temp, persp in VOTE_CFG:
                msg = [{"role": "system", "content": system_prompt(lg, persp)},
                       {"role": "user", "content": f"{FEW_BLOCK}\n\nNOW CLASSIFY:\nClause: {t[:1200]}\n\nRetrieved clauses:\n{ctx_txt}\n\nOutput ONLY: {{\"overall_risk\": \"none|low|medium|high\"}}"}]
                v.append(parse_risk(chat(MAIN_MODEL, msg, temp)))
            votes.append(v)
        p3, pb, cf = aggregate(votes, k=1, use_kw=True)
        rag_runs.append({"run": run + 1, "macro3": macro3(y_ext3, p3), "binary_f1": binf1(y_extb, pb),
                         "acc_bin": accuracy_score(y_extb, pb), "minutes": (time.time() - t_start) / 60,
                         "votes": votes, "p3": p3, "pb": pb, "conf": cf})
        pd.DataFrame({"passage_id": ev["passage_id"], "votes": [str(x) for x in votes]}).to_excel(
            f"{OUT}/rag_run{run+1}_votes.xlsx", sheet_name="v3_predictions", index=False)
        print(f"Run {run+1}: macro-F1={rag_runs[-1]['macro3']:.3f} binary F1={rag_runs[-1]['binary_f1']:.3f}")
    bf = [r["binary_f1"] for r in rag_runs]; m3 = [r["macro3"] for r in rag_runs]
    print(f"\nBinary F1 sur {N_REPEATS} runs : {np.mean(bf):.3f} ± {np.std(bf, ddof=1):.3f} (min {min(bf):.3f}, max {max(bf):.3f})")
    print(f"3-class macro-F1 : {np.mean(m3):.3f} ± {np.std(m3, ddof=1):.3f}")
    # stabilité vote à vote entre runs
    agree = np.mean([rag_runs[0]["pb"][i] == rag_runs[j]["pb"][i] for j in range(1, N_REPEATS) for i in range(len(ev))])
    print(f"Accord des prédictions binaires run 1 vs autres runs : {agree:.3f}")
    record("ClauseRisk RAG (run 1, re-impl.)", None, rag_runs[0]["p3"], predb=rag_runs[0]["pb"], conf=rag_runs[0]["conf"])
    npass = len(ev) * N_REPEATS; cpp = []
    for m in USAGE:
        d = {k: USAGE[m][k] - U0.get(m, {}).get(k, 0) for k in ("in", "out", "calls", "sec")}
        pi, po = PRICES[m]
        cpp.append({"model": m, "calls_per_passage": d["calls"] / npass, "tokens_in_per_passage": d["in"] / npass,
                    "tokens_out_per_passage": d["out"] / npass, "seconds_per_passage": d["sec"] / npass,
                    "usd_per_passage": (d["in"] * pi + d["out"] * po) / 1e6 / npass})
    COST_PP = pd.DataFrame(cpp)
    print("\n=== Coût par passage (pipeline RAG : 1 HyDE + 5 votes GPT-4o, appels séquentiels) ===")
    print(COST_PP.round(4).to_string(index=False))
    print(f"TOTAL par passage : {COST_PP['usd_per_passage'].sum():.4f} USD, {COST_PP['seconds_per_passage'].sum():.1f} s")

if RUN_LLM_ONLY or RUN_RAG_REPEATS:
    n_pass = len(ev) * ((1 if RUN_LLM_ONLY else 0) + (N_REPEATS if RUN_RAG_REPEATS else 0))
    tot = sum(cost(m) for m in USAGE)
    print("\n=== Coût ===")
    for m, u in USAGE.items(): print(f"{m}: {u['calls']} appels, {u['in']} tokens in, {u['out']} out, {cost(m):.3f} USD")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
Run 1: macro-F1=0.689 binary F1=0.827
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
Run 2: macro-F1=0.718 binary F1=0.838
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
  retry 0 Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-MqIiTmtC4gCBaFGMj76S2xtl on t
  retry

In [13]:
# ---------------- 11. v5 STOCKÉ, COUVERTURE ÉGALE, McNEMAR ----------------
if V3_FILE:
    dv = None
    for s in pd.ExcelFile(V3_FILE).sheet_names:
        d = pd.read_excel(V3_FILE, sheet_name=s)
        if "votes" in d.columns: dv = d; break
    dv["votes_l"] = dv["votes"].apply(lambda s: re.findall(r"(none|low|medium|high)", str(s).lower()))
    vmap = dict(zip(dv["passage_id"], dv["votes_l"]))
    v5_votes = [vmap.get(pid, []) for pid in ev["passage_id"]]
    if "KW" not in globals():
        raise RuntimeError("Exécuter d'abord la cellule 9 (veto) — ou mettre RUN_LLM_ONLY=True")
    p3, pb, cf = aggregate(v5_votes, k=1, use_kw=True)
    record("ClauseRisk v5 (stored votes)", None, p3, predb=pb, conf=cf)
    assert abs(binf1(y_extb, pb) - 0.806) < 0.002, "v5 ne retrouve pas 0.806 : vérifier le fichier de votes"

def selective(name, cov):
    r = RESULTS[name]
    if r["conf"] is None: return None
    m = int(round(cov * len(ev))); order = np.lexsort((np.random.default_rng(SEED).random(len(ev)), -r["conf"]))[:m]
    return accuracy_score(y_extb[order], r["predb"][order]), f1_score(y_extb[order], r["predb"][order], zero_division=0)
eq = []
for name in RESULTS:
    for cov in [0.5, 0.7, 0.9, 1.0]:
        s = selective(name, cov)
        if s: eq.append({"system": name, "coverage": cov, "selective_acc": round(s[0], 3), "selective_F1": round(s[1], 3)})
df_eq = pd.DataFrame(eq)
print(df_eq.pivot(index="system", columns="coverage", values="selective_acc").round(3))

ref = "ClauseRisk v5 (stored votes)" if "ClauseRisk v5 (stored votes)" in RESULTS else None
mc = []
if ref:
    for name, r in RESULTS.items():
        if name == ref: continue
        a, b, p = mcnemar(y_extb, RESULTS[ref]["predb"], r["predb"])
        mc.append({"A": ref, "B": name, "A right/B wrong": a, "A wrong/B right": b, "p_exact": round(p, 4)})
df_mc = pd.DataFrame(mc); print(df_mc.to_string(index=False))

ClauseRisk v5 (stored votes)       int=None  ext3=0.68  extB=0.806
coverage                            0.5    0.7    0.9    1.0
system                                                      
ClauseRisk RAG (run 1, re-impl.)  0.875  0.877  0.863  0.840
ClauseRisk v5 (stored votes)      0.875  0.877  0.849  0.827
GPT-4o without retrieval          0.850  0.860  0.822  0.778
MiniLM embeddings + LogReg        0.575  0.579  0.562  0.556
XLM-R-base, fine-tuned            0.500  0.456  0.507  0.531
mE5 k-NN (no generation)          0.550  0.561  0.548  0.556
                           A                                B  A right/B wrong  A wrong/B right  p_exact
ClauseRisk v5 (stored votes)                         Majority               29                6   0.0001
ClauseRisk v5 (stored votes)                           Random               30                4   0.0000
ClauseRisk v5 (stored votes)                  TF-IDF + LogReg               33                6   0.0000
ClauseRisk v5 (stored vot

In [14]:
# ---------------- 12. LIGNES LaTeX + EXPORT ----------------
def f(x): return "---" if x is None else f"{x:.3f}"
order = ["Majority", "Random", "TF-IDF + LogReg", "TF-IDF + SVM", "TF-IDF + RF", "TF-IDF + GB",
         "MiniLM embeddings + LogReg", "XLM-R-base, fine-tuned", "mE5 k-NN (no generation)", "BM25 + Rule",
         "GPT-4o without retrieval"]
print("% ---- Lignes pour le Tableau 12 (tab:internal_external) ----")
for n in order:
    if n not in RESULTS: continue
    r = RESULTS[n]; d = None if r["internal_macro"] is None or r["external_macro"] is None else r["external_macro"] - r["internal_macro"]
    lo, hi = boot_ci(y_extb, r["predb"])
    name = n.replace("k-NN", "$k$-NN")
    print(f"{name} & {f(r['internal_macro'])} & {f(r['external_macro'])} & {'---' if d is None else f'${d:+.3f}$'} & {r['external_binary']:.3f} [{lo:.3f}--{hi:.3f}] \\\\")
if "XLM-R-base, fine-tuned" in RESULTS:
    r = RESULTS["XLM-R-base, fine-tuned"]
    print(f"% XLM-R par seed : interne {np.mean(r['seed_internal']):.3f} ± {np.std(r['seed_internal']):.3f}, "
          f"externe {np.mean(r['seed_external']):.3f} ± {np.std(r['seed_external']):.3f}, binaire {np.mean(r['seed_binary']):.3f} ± {np.std(r['seed_binary']):.3f}")
if RUN_RAG_REPEATS:
    print(f"% Coût par passage : {COST_PP['tokens_in_per_passage'].sum():.0f} tokens in, {COST_PP['tokens_out_per_passage'].sum():.0f} out, {COST_PP['seconds_per_passage'].sum():.1f} s, {COST_PP['usd_per_passage'].sum():.4f} USD")
    print(f"% Runs répétés : binary F1 {np.mean(bf):.3f} $\\pm$ {np.std(bf, ddof=1):.3f} (range {min(bf):.3f}--{max(bf):.3f}), accord binaire {agree:.3f}")

rows = [{"method": n, "internal_macro": r["internal_macro"], "external_macro": r["external_macro"],
         "external_binary": r["external_binary"]} for n, r in RESULTS.items()]
with pd.ExcelWriter(f"{OUT}/baseline_results.xlsx") as xw:
    pd.DataFrame(rows).to_excel(xw, sheet_name="summary", index=False)
    df_eq.to_excel(xw, sheet_name="equal_coverage", index=False)
    if len(df_mc): df_mc.to_excel(xw, sheet_name="mcnemar_vs_v5", index=False)
    if RUN_RAG_REPEATS:
        pd.DataFrame([{k: v for k, v in r.items() if k in ("run", "macro3", "binary_f1", "acc_bin", "minutes")} for r in rag_runs]).to_excel(xw, sheet_name="repeated_runs", index=False)
        COST_PP.to_excel(xw, sheet_name="cost_per_passage", index=False)
    if RUN_LLM_ONLY or RUN_RAG_REPEATS:
        pd.DataFrame([{"model": m, **u, "usd": cost(m)} for m, u in USAGE.items()]).to_excel(xw, sheet_name="api_usage", index=False)
    pd.DataFrame({"passage_id": ev["passage_id"], "gold": y_ext3,
                  **{f"pred_{n}": r["predb"] for n, r in RESULTS.items()}}).to_excel(xw, sheet_name="binary_predictions", index=False)
import shutil; shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files; files.download(f"{OUT}.zip")
except ImportError: pass
print("Terminé ->", OUT + ".zip")

% ---- Lignes pour le Tableau 12 (tab:internal_external) ----
Majority & 0.262 & 0.121 & $-0.140$ & 0.000 [0.000--0.000] \\
Random & 0.333 & 0.338 & $+0.004$ & 0.394 [0.242--0.526] \\
TF-IDF + LogReg & 0.809 & 0.179 & $-0.629$ & 0.000 [0.000--0.000] \\
TF-IDF + SVM & 0.786 & 0.179 & $-0.608$ & 0.000 [0.000--0.000] \\
TF-IDF + RF & 0.767 & 0.126 & $-0.641$ & 0.000 [0.000--0.000] \\
TF-IDF + GB & 0.744 & 0.147 & $-0.598$ & 0.000 [0.000--0.000] \\
MiniLM embeddings + LogReg & 0.751 & 0.250 & $-0.500$ & 0.100 [0.000--0.233] \\
XLM-R-base, fine-tuned & 0.865 & 0.224 & $-0.640$ & 0.000 [0.000--0.000] \\
mE5 $k$-NN (no generation) & --- & 0.170 & --- & 0.053 [0.000--0.150] \\
BM25 + Rule & --- & 0.236 & --- & 0.388 [0.237--0.526] \\
GPT-4o without retrieval & --- & 0.746 & --- & 0.775 [0.684--0.861] \\
% XLM-R par seed : interne 0.865 ± 0.044, externe 0.207 ± 0.017, binaire 0.033 ± 0.047
% Coût par passage : 5530 tokens in, 96 out, 10.5 s, 0.0140 USD
% Runs répétés : binary F1 0.823 $\pm$ 0.0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Terminé -> revision_experiments.zip
